# Exposure at Default (EAD) Forecasting

### Credit Scoring Risk Assessment — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of credit scoring and risk assessment terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the credit scoring and risk assessment prediction task and why regression modelling supports this banking decision.
- Implement a regression workflow and evaluate predictive accuracy using domain-appropriate error metrics.
- Evaluate whether the prediction error is acceptable for the operational decision it supports.

**Estimated time:** 35–45 minutes

**Collection context:** Credit scoring, probability of default, loss estimation, and stress testing for banking risk management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** For a fixed loan (like a mortgage), EAD is simple—it's just the remaining balance. But for credit cards or business lines of credit, the borrower might "draw down" more money as they get closer to defaulting. If someone has a \$10k limit and has used \$2k, but is about to go bankrupt, they might quickly spend the remaining \$8k. EAD forecasting predicts this "extra" usage.

**Input data used:** Current balance, credit limit, time since last increase, utilization rate, and payment history.

**What we predict:** Credit Conversion Factor (CCF) — what % of the *unused* limit will be used by the time of default.

**ML method used:** Random Forest Regressor.

**Learning goal:** Learn how to model contingent liabilities (money the bank hasn't lent yet but might have to).

**Primary stakeholders:** credit analysts, loan officers, risk managers, and regulatory auditors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Credit Line Dataset

We generate 2,500 credit card accounts that eventually defaulted.

In [ ]:
n = 2500
rng = RNG

credit_limit = rng.choice([1000, 5000, 10000, 25000], n)
current_utilization = rng.uniform(0.1, 0.8, n) # Utilization at T-6 months
current_balance = credit_limit * current_utilization
unused_limit = credit_limit - current_balance

months_since_active = rng.integers(0, 12, n)
past_overlimit_events = rng.poisson(0.2, n)

# Logic for CCF (Credit Conversion Factor)
# CCF = (Balance_at_Default - Current_Balance) / Unused_Limit
ccf = (
    0.3 + 
    0.4 * current_utilization + 
    0.1 * past_overlimit_events + 
    rng.normal(0, 0.1, n)
).clip(0, 1)

df = pd.DataFrame({
    'credit_limit': credit_limit,
    'current_balance': current_balance,
    'unused_limit': unused_limit,
    'current_utilization': current_utilization,
    'months_since_active': months_since_active,
    'past_overlimit': past_overlimit_events,
    'actual_ccf': ccf
})

print("Sample of credit line data:")
print(df.head())

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
# Features are used as created in Step 3.
print("Target column: 'actual_ccf'")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the training-set mean ---
baseline_reg = DummyRegressor(strategy='mean')
baseline_reg.fit(X_train, y_train)
baseline_pred_bl = baseline_reg.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred_bl))
print(f"Baseline RMSE (predict-mean): {baseline_rmse:.4f}")
print("Any useful model must produce a lower RMSE.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

We train a model to predict the CCF based on current behavior.

In [ ]:
X = df[['credit_limit', 'current_balance', 'current_utilization', 'months_since_active', 'past_overlimit']]
y = df['actual_ccf']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X_train, y_train)

predicted_ccf = model.predict(X_test)

In [ ]:
# EAD = Current Balance + (Predicted CCF * Unused Limit)
predicted_ead = X_test['current_balance'] + (predicted_ccf * X_test['unused_limit'])
actual_ead = X_test['current_balance'] + (y_test * X_test['unused_limit'])

plt.figure(figsize=(10, 6))
plt.scatter(actual_ead, predicted_ead, alpha=0.5, color='#264653')
plt.plot([0, 25000], [0, 25000], '--r')
plt.xlabel('Actual Exposure at Default (\$)')
plt.ylabel('Predicted Exposure at Default (\$)')
plt.title('Actual vs. Predicted EAD')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot, line chart titled "Actual vs. Predicted EAD" with Actual Exposure at Default (\$) on the x-axis and Predicted Exposure at Default (\$) on the y-axis. The chart highlights spatial separation or clustering patterns in the data.

In [ ]:
total_current = X_test['current_balance'].sum()
total_ead_forecast = predicted_ead.sum()

print(f"Total Current Balance in Test Set: \${total_current:,.0f}")
print(f"Total Predicted EAD in Test Set:   \${total_ead_forecast:,.0f}")
print(f"Predicted 'Drawdown' at Default:   \${(total_ead_forecast - total_current):,.0f} (+{((total_ead_forecast/total_current)-1)*100:.1f}%)")

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- Borrowers with **high current utilization** and **past overlimit events** are predicted to have higher CCFs (they use more of their remaining limit before defaulting).
- The EAD forecast shows that the bank is actually exposed to ~20-30% more money than is currently showing on the balance sheets for these high-risk accounts.

**Banking Context:** EAD forecasting is critical for credit card issuers. It prevents the bank from underestimating its risk by only looking at today's balances.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: inspect predictions at different points ---
print("Operational demonstration — model predictions across the test range:\n")
low_idx  = int(np.argmin(y_test.values))
high_idx = int(np.argmax(y_test.values))
mid_idx  = int(np.argsort(y_test.values)[len(y_test) // 2])

for label, idx in [("Low-end", low_idx), ("Mid-range", mid_idx), ("High-end", high_idx)]:
    actual = y_test.values[idx]
    pred   = y_pred[idx]
    print(f"  {label}  actual: {actual:.4f}  predicted: {pred:.4f}  error: {abs(actual-pred):.4f}")

print("\nPractitioners would use these predictions as one input among several.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end credit scoring and risk assessment workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Credit models can perpetuate historical discrimination if trained on biased lending data.
- Proxy variables such as zip code or employment type may correlate with protected characteristics.
- Applicants deserve transparent explanations of adverse credit decisions under fair lending regulations.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in credit scoring and risk assessment?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.